# Exploratory Data Analysis — Retail Rocket E-Commerce Dataset

This notebook explores the Retail Rocket dataset to understand:
- User behavioral patterns (views, add-to-cart, purchases)
- Item popularity distributions
- Temporal trends
- Data sparsity and cold-start challenges

**Dataset**: [Retail Rocket](https://www.kaggle.com/retailrocket/ecommerce-dataset) — real e-commerce behavioral data

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('seaborn-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

%matplotlib inline

## 1. Load Raw Data

In [ ]:
import os
PROJECT_ROOT = os.path.abspath('..')
events = pd.read_csv(os.path.join(PROJECT_ROOT, 'data/raw/events.csv'))
print(f"Events shape: {events.shape}")
print(f"Columns: {list(events.columns)}")
events.head()

In [ ]:
events['datetime'] = pd.to_datetime(events['timestamp'], unit='ms')
events['date'] = events['datetime'].dt.date
events['hour'] = events['datetime'].dt.hour
events['day_of_week'] = events['datetime'].dt.dayofweek

print(f"Date range: {events['datetime'].min()} to {events['datetime'].max()}")
print(f"\nBasic statistics:")
print(f"  Total events: {len(events):,}")
print(f"  Unique users: {events['visitorid'].nunique():,}")
print(f"  Unique items: {events['itemid'].nunique():,}")
events.info()

## 2. Event Distribution Analysis

In [ ]:
event_counts = events['event'].value_counts()
event_pcts = events['event'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#3498db', '#e74c3c', '#2ecc71']
event_counts.plot(kind='bar', ax=axes[0], color=colors)
axes[0].set_title('Event Type Distribution (Count)')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

for i, (count, pct) in enumerate(zip(event_counts, event_pcts)):
    axes[0].text(i, count + count*0.02, f'{count:,}\n({pct:.1f}%)',
                 ha='center', va='bottom', fontweight='bold')

funnel_data = pd.DataFrame({
    'stage': ['View', 'Add to Cart', 'Purchase'],
    'users': [
        events[events['event']=='view']['visitorid'].nunique(),
        events[events['event']=='addtocart']['visitorid'].nunique(),
        events[events['event']=='transaction']['visitorid'].nunique()
    ]
})
axes[1].barh(funnel_data['stage'], funnel_data['users'], color=colors)
axes[1].set_title('Conversion Funnel (Unique Users)')
axes[1].set_xlabel('Number of Users')
for i, v in enumerate(funnel_data['users']):
    axes[1].text(v + v*0.02, i, f'{v:,}', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\nConversion rates:")
print(f"  View → Cart: {funnel_data['users'].iloc[1]/funnel_data['users'].iloc[0]*100:.2f}%")
print(f"  Cart → Purchase: {funnel_data['users'].iloc[2]/funnel_data['users'].iloc[1]*100:.2f}%")
print(f"  View → Purchase: {funnel_data['users'].iloc[2]/funnel_data['users'].iloc[0]*100:.2f}%")

## 3. User Behavior Analysis

In [ ]:
user_event_counts = events.groupby('visitorid')['event'].count()
user_item_counts = events.groupby('visitorid')['itemid'].nunique()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

user_event_counts.clip(upper=user_event_counts.quantile(0.99)).hist(
    bins=50, ax=axes[0], color='#3498db', edgecolor='white'
)
axes[0].set_title('Distribution of Events per User (99th percentile)')
axes[0].set_xlabel('Number of Events')
axes[0].set_ylabel('Number of Users')
axes[0].axvline(user_event_counts.median(), color='red', linestyle='--',
                label=f'Median: {user_event_counts.median():.0f}')
axes[0].legend()

user_item_counts.clip(upper=user_item_counts.quantile(0.99)).hist(
    bins=50, ax=axes[1], color='#2ecc71', edgecolor='white'
)
axes[1].set_title('Distribution of Unique Items per User (99th percentile)')
axes[1].set_xlabel('Number of Unique Items')
axes[1].set_ylabel('Number of Users')
axes[1].axvline(user_item_counts.median(), color='red', linestyle='--',
                label=f'Median: {user_item_counts.median():.0f}')
axes[1].legend()

plt.tight_layout()
plt.show()

print("User activity statistics:")
print(user_event_counts.describe().to_string())

## 4. Item Popularity Analysis

In [ ]:
item_event_counts = events.groupby('itemid')['event'].count().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

top_n = 30
item_event_counts.head(top_n).plot(kind='bar', ax=axes[0], color='#e74c3c')
axes[0].set_title(f'Top {top_n} Most Popular Items')
axes[0].set_xlabel('Item ID')
axes[0].set_ylabel('Number of Events')
axes[0].tick_params(axis='x', rotation=45)

cumulative = item_event_counts.cumsum() / item_event_counts.sum() * 100
pct_items = np.arange(1, len(cumulative) + 1) / len(cumulative) * 100
axes[1].plot(pct_items, cumulative.values, color='#9b59b6', linewidth=2)
axes[1].axhline(y=80, color='red', linestyle='--', alpha=0.7, label='80% of events')
idx_80 = np.searchsorted(cumulative.values, 80)
pct_80 = pct_items[idx_80] if idx_80 < len(pct_items) else pct_items[-1]
axes[1].axvline(x=pct_80, color='red', linestyle='--', alpha=0.7)
axes[1].set_title('Long-Tail Distribution (Pareto)')
axes[1].set_xlabel('% of Items')
axes[1].set_ylabel('% of Total Events (Cumulative)')
axes[1].legend([f'80% of events from {pct_80:.1f}% of items'])

plt.tight_layout()
plt.show()

print(f"\nItem popularity statistics:")
print(item_event_counts.describe().to_string())

## 5. Temporal Patterns

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

daily = events.groupby(['date', 'event']).size().unstack(fill_value=0)
daily.plot(ax=axes[0, 0], linewidth=1.5)
axes[0, 0].set_title('Daily Events by Type')
axes[0, 0].set_ylabel('Count')
axes[0, 0].legend(loc='upper right')

hourly = events.groupby(['hour', 'event']).size().unstack(fill_value=0)
hourly.plot(ax=axes[0, 1], linewidth=2)
axes[0, 1].set_title('Hourly Event Patterns')
axes[0, 1].set_xlabel('Hour of Day')
axes[0, 1].set_ylabel('Count')

dow_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
dow = events.groupby(['day_of_week', 'event']).size().unstack(fill_value=0)
dow.index = [dow_names[i] for i in dow.index]
dow.plot(kind='bar', ax=axes[1, 0])
axes[1, 0].set_title('Events by Day of Week')
axes[1, 0].tick_params(axis='x', rotation=0)

daily_users = events.groupby('date')['visitorid'].nunique()
daily_users.plot(ax=axes[1, 1], color='#3498db', linewidth=1.5)
axes[1, 1].set_title('Daily Active Users')
axes[1, 1].set_ylabel('Unique Users')

plt.tight_layout()
plt.show()

## 6. Sparsity & Cold-Start Analysis

In [ ]:
n_users = events['visitorid'].nunique()
n_items = events['itemid'].nunique()
n_interactions = events.drop_duplicates(subset=['visitorid', 'itemid']).shape[0]
sparsity = 1 - (n_interactions / (n_users * n_items))

print(f"=== Interaction Matrix Statistics ===")
print(f"Users: {n_users:,}")
print(f"Items: {n_items:,}")
print(f"Possible interactions: {n_users * n_items:,}")
print(f"Actual interactions: {n_interactions:,}")
print(f"Sparsity: {sparsity:.6%}")
print(f"Density: {1-sparsity:.6%}")

print(f"\n=== Cold-Start Analysis ===")
user_counts = events.groupby('visitorid')['itemid'].nunique()
item_counts = events.groupby('itemid')['visitorid'].nunique()

for threshold in [1, 2, 3, 5, 10]:
    cold_users = (user_counts <= threshold).sum()
    cold_items = (item_counts <= threshold).sum()
    print(f"  Users with ≤{threshold} items: {cold_users:,} ({cold_users/n_users*100:.1f}%)")
    print(f"  Items with ≤{threshold} users: {cold_items:,} ({cold_items/n_items*100:.1f}%)")

## 7. Data Preprocessing Preview

In [ ]:
from src.data_loader import RetailRocketDataLoader

loader = RetailRocketDataLoader()
loader.run_pipeline()

print(f"\nAfter preprocessing:")
print(f"  Interactions: {len(loader.interactions):,}")
print(f"  Users: {loader.interactions['visitorid'].nunique():,}")
print(f"  Items: {loader.interactions['itemid'].nunique():,}")
print(f"\nRating distribution:")
print(loader.interactions['rating'].describe())

loader.interactions['rating'].hist(bins=50, figsize=(10, 4), color='#3498db', edgecolor='white')
plt.title('Implicit Rating Distribution')
plt.xlabel('Rating')
plt.ylabel('Count')
plt.show()

In [ ]:
train, val, test = loader.temporal_train_test_split()

print(f"Train: {len(train):,} ({len(train)/len(loader.interactions)*100:.1f}%)")
print(f"Val:   {len(val):,} ({len(val)/len(loader.interactions)*100:.1f}%)")
print(f"Test:  {len(test):,} ({len(test)/len(loader.interactions)*100:.1f}%)")

print(f"\nUser overlap (train→test): "
      f"{len(set(train['visitorid']) & set(test['visitorid'])):,} / {test['visitorid'].nunique():,}")
print(f"Item overlap (train→test): "
      f"{len(set(train['itemid']) & set(test['itemid'])):,} / {test['itemid'].nunique():,}")

## Key Takeaways

Document findings here after running the EDA:

1. **Data Scale**: [Users] x [Items] with [Sparsity]% sparsity
2. **Long-tail**: Most events concentrated on small % of items → need popularity bias handling
3. **Cold-start**: Significant portion of users/items with very few interactions
4. **Conversion funnel**: View → Cart → Purchase conversion rates
5. **Temporal patterns**: Activity peaks and weekly patterns inform real-time features